# TrialScreen: full pipeline on a free Kaggle GPU

Run this notebook top to bottom on **Kaggle** (Settings: Accelerator = GPU T4, Internet = On).
It trains, evaluates, benchmarks and produces every table/figure used in the report.
Total time on one T4 is roughly 60 to 90 minutes. Outputs end up in `/kaggle/working/trialscreen_outputs.zip`.

The final application does **not** run from this notebook: it is a browser app (see README).

## 1. Get the code
Option A (recommended): clone your GitHub repository. Option B: upload `trialscreen.zip` as a Kaggle Dataset and copy it.

In [ ]:
REPO = "https://github.com/<your-username>/trialscreen.git"   # <-- change
import os, subprocess
if not os.path.exists("/kaggle/working/trialscreen"):
    if "<your-username>" not in REPO:
        subprocess.run(["git", "clone", REPO, "/kaggle/working/trialscreen"], check=True)
    else:  # option B: zip uploaded as a dataset named 'trialscreen'
        subprocess.run("cp -r /kaggle/input/trialscreen/trialscreen /kaggle/working/ || unzip -q /kaggle/input/trialscreen/trialscreen.zip -d /kaggle/working/", shell=True, check=True)
%cd /kaggle/working/trialscreen
!ls

In [ ]:
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0" scikit-learn matplotlib pytest
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data: fixed 37/8/8 patient split (seed 42) + leakage checks + unit tests

In [ ]:
!python scripts/01_prepare_data.py
!python -m pytest -q tests | tail -3

## 3. Fine-tune Qwen2.5-1.5B-Instruct with LoRA (train split; model selection on validation macro-F1)

In [ ]:
!python scripts/02_train_lora.py

## 4. Before/after evaluation
Validation first. The test split is evaluated **once**, after all choices are frozen.

In [ ]:
!python scripts/03_evaluate.py --split validation

In [ ]:
!python scripts/03_evaluate.py --split test --final

## 5. Task 3: KV-cache benchmark (3 input x 3 output lengths, warm-up, 3 synchronized repeats)

In [ ]:
!python scripts/04_bench_kv.py

## 6. Task 4: tiled online-softmax attention, correctness and performance

In [ ]:
!python scripts/05_attention.py

## 7. Task 5: greedy speculative decoding (draft Qwen2.5-0.5B-Instruct)

In [ ]:
!python scripts/06_bench_speculative.py

## 8. Tables, figures and the error-analysis worksheet for the report

In [ ]:
!python scripts/07_make_report_assets.py
!ls report/figures report/tables

## 9. Publish the adapter (needed by the app)
Add a Kaggle secret `HF_TOKEN` (a Hugging Face write token), then run:

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO = "<your-hf-username>/qwen2.5-1.5b-trialscreen-lora"   # <-- change
!python scripts/08_push_adapter.py --repo {HF_REPO}

## 10. Download everything

In [ ]:
!zip -qr /kaggle/working/trialscreen_outputs.zip results report checkpoints/qwen2.5-1.5b-trialscreen-lora/best_adapter data/splits
!ls -lh /kaggle/working/trialscreen_outputs.zip